# GroupIndex Benchmark

Benchmark different GroupIndex implementations across varying dataset sizes.

Outer loop: dataset sizes (5000, 10000, 20000, 25000, 50000, 75000, 100000, 133150)
Inner loop: GroupIndex implementations (lazy, dict, hash, pure, numba)

Each run computes 20 approximate reducts with 30 candidates using greedy heuristic.
If a run exceeds 3 minutes, that GroupIndex implementation is skipped for all larger sizes.

Results are saved to a JSON file after each dataset size iteration.

In [1]:
import json
import pathlib
import time

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from skrough.algorithms.reducts import get_approx_reduct_greedy_heuristic
from skrough.disorder_measures import gini_impurity

In [4]:
# Configuration

DATA_DIR = pathlib.Path("data")
OUTPUT_FILE = pathlib.Path("group_index_benchmark_results.json")

DATASET_SIZES = [1000, 2000, 5000, 10000, 20000, 25000, 50000, 75000, 100000, 133150]
GROUP_INDEX_ORDER = ["lazy", "dict", "hash", "pure", "numba"]

CANDIDATES_COUNT = 20
N_REDUCTS = 10
N_JOBS = -1
EPSILON = 0.2

TIMEOUT_SECONDS = 120  # 2 minutes

In [5]:
# Load and encode the full dataset

data = pd.read_csv(DATA_DIR / "seismic_training_data_features_discretized.csv")
target = pd.read_csv(DATA_DIR / "seismic_training_labels.csv", header=None)[0]

data.drop("main_working_id", axis=1, inplace=True)

data_encoded = OrdinalEncoder().fit_transform(data)
target_encoded = LabelEncoder().fit_transform(target)

print(f"Full dataset shape: {data_encoded.shape}")
print(f"Total objects: {len(data_encoded)}")

Full dataset shape: (133151, 738)
Total objects: 133151


In [6]:
# Benchmark runner

results = {}
skipped = set()  # GroupIndex implementations skipped due to timeout

for size in DATASET_SIZES:
    print(f"\n{'=' * 60}")
    print(f"Dataset size: {size}")
    print(f"{'=' * 60}")

    actual_size = min(size, len(data_encoded))
    if actual_size < size:
        print(
            f"  Note: requested size {size} exceeds available data, using {actual_size}"
        )

    x_subset, _, y_subset, _ = train_test_split(
        data_encoded, target_encoded, train_size=actual_size, random_state=42
    )
    print(f"  Actual subset size: {len(x_subset)}")

    results[size] = {}

    for gi_name in GROUP_INDEX_ORDER:
        if gi_name in skipped:
            print(f"  [{gi_name}] SKIPPED (timed out at smaller size)")
            results[size][gi_name] = {
                "status": "skipped",
                "reason": "timeout_at_smaller_size",
            }
            continue

        print(f"  [{gi_name}] Running...", end=" ", flush=True)
        start = time.perf_counter()

        try:
            get_approx_reduct_greedy_heuristic(
                x=x_subset,
                y=y_subset,
                disorder_fun=gini_impurity,
                epsilon=EPSILON,
                candidates_count=CANDIDATES_COUNT,
                n_reducts=N_REDUCTS,
                n_jobs=N_JOBS,
                group_index_class=gi_name,
            )
            elapsed = time.perf_counter() - start
            print(f"{elapsed:.2f}s")

            results[size][gi_name] = {
                "status": "completed",
                "time_seconds": round(elapsed, 3),
                "actual_size": actual_size,
            }

            if elapsed > TIMEOUT_SECONDS:
                print(
                    f"  [{gi_name}] TIMEOUT ({elapsed:.2f}s > {TIMEOUT_SECONDS}s), skipping larger sizes"
                )
                skipped.add(gi_name)

        except Exception as e:
            elapsed = time.perf_counter() - start
            print(f"ERROR ({elapsed:.2f}s): {e}")
            results[size][gi_name] = {
                "status": "error",
                "time_seconds": round(elapsed, 3),
                "error": str(e),
            }

    # Save results after each size iteration
    with open(OUTPUT_FILE, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n  Results saved to {OUTPUT_FILE}")


Dataset size: 1000
  Actual subset size: 1000
  [lazy] Running... 25.49s
  [dict] Running... 0.83s
  [hash] Running... 1.01s
  [pure] Running... 0.46s
  [numba] Running... 0.25s

  Results saved to group_index_benchmark_results.json

Dataset size: 2000
  Actual subset size: 2000
  [lazy] Running... 49.75s
  [dict] Running... 1.12s
  [hash] Running... 1.22s
  [pure] Running... 0.14s
  [numba] Running... 0.26s

  Results saved to group_index_benchmark_results.json

Dataset size: 5000
  Actual subset size: 5000
  [lazy] Running... 124.84s
  [lazy] TIMEOUT (124.84s > 120s), skipping larger sizes
  [dict] Running... 3.29s
  [hash] Running... 2.97s
  [pure] Running... 0.19s
  [numba] Running... 0.30s

  Results saved to group_index_benchmark_results.json

Dataset size: 10000
  Actual subset size: 10000
  [lazy] SKIPPED (timed out at smaller size)
  [dict] Running... 8.29s
  [hash] Running... 6.10s
  [pure] Running... 0.27s
  [numba] Running... 0.41s

  Results saved to group_index_benchmark

In [7]:
# Summary table

import pandas as pd

summary_rows = []
for size in DATASET_SIZES:
    for gi_name in GROUP_INDEX_ORDER:
        entry = results.get(size, {}).get(gi_name, {})
        summary_rows.append(
            {
                "dataset_size": size,
                "group_index": gi_name,
                "status": entry.get("status", "not_run"),
                "time_seconds": entry.get("time_seconds", None),
            }
        )

summary_df = pd.DataFrame(summary_rows)
pivot = summary_df.pivot(
    index="dataset_size", columns="group_index", values="time_seconds"
)
print("\nTime (seconds) by dataset size and GroupIndex implementation:")
print(pivot.to_string())


Time (seconds) by dataset size and GroupIndex implementation:
group_index      dict    hash     lazy  numba   pure
dataset_size                                        
1000            0.834   1.015   25.489  0.250  0.461
2000            1.121   1.219   49.747  0.264  0.144
5000            3.288   2.975  124.835  0.298  0.189
10000           8.287   6.104      NaN  0.413  0.272
20000          21.017  12.466      NaN  0.571  0.433
25000          28.282  16.033      NaN  0.606  0.533
50000          63.995  35.101      NaN  1.170  1.100
75000         100.361  52.955      NaN  2.024  2.181
100000        144.850  74.444      NaN  3.084  3.221
133150            NaN  99.452      NaN  4.763  4.933


In [ ]:
# Plot: time vs. dataset size for each GroupIndex implementation

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

for gi_name in GROUP_INDEX_ORDER:
    sizes = []
    times = []
    for size in DATASET_SIZES:
        entry = results.get(size, {}).get(gi_name, {})
        if entry.get("status") == "completed":
            sizes.append(entry.get("actual_size", size))
            times.append(entry["time_seconds"])
    if sizes:
        ax.plot(sizes, times, marker="o", label=gi_name)

ax.set_xlabel("Number of objects")
ax.set_ylabel("Time (seconds)")
ax.set_title("GroupIndex Benchmark: Time vs. Dataset Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("group_index_benchmark_plot.png", dpi=150)
plt.show()